<a href="https://colab.research.google.com/github/preranasinha20/ato/blob/main/PBL_MT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import pandas as pd

df = pd.read_csv('/content/siem_logs_only.csv')

print(df.shape)
df.head()

(100000, 2)


,raw_log,isFraud
0,[2026-01-12 14:00:00] EVENT=TRANSACTION USER=C...,0
1,[2026-01-01 15:00:00] EVENT=TRANSACTION USER=C...,0
2,[2026-01-01 10:00:00] EVENT=TRANSACTION USER=C...,0
3,[2026-01-17 19:00:00] EVENT=TRANSACTION USER=C...,0
4,[2026-01-09 14:00:00] EVENT=TRANSACTION USER=C...,0


In [45]:
import re

def parse_log(log):
    data = {}

    patterns = {
        'timestamp': r'\[(.*?)\]',
        'transaction_type': r'ACTION=(\w+)',
        'amount': r'AMOUNT=(\d+\.?\d*)',
        'device_type': r'DEVICE=(\w+)',
        'os': r'OS=(\w+)',
        'browser': r'BROWSER=(\w+)',
        'ip_address': r'IP=([\d\.]+)',
        'ip_city': r'CITY=(\w+)',
        'login_method': r'LOGIN_METHOD=(\w+)',
        'failed_login_count': r'FAILED_LOGINS=(\d+)',
        'is_new_device': r'NEW_DEVICE=(\d+)',
        'event_velocity': r'VELOCITY=(\d+\.?\d*)'
    }

    for key, pattern in patterns.items():
        match = re.search(pattern, log)
        data[key] = match.group(1) if match else None

    return data


In [46]:
parsed = df['raw_log'].apply(parse_log)
parsed_df = pd.DataFrame(parsed.tolist())

parsed_df.head()

,timestamp,transaction_type,amount,device_type,os,browser,ip_address,ip_city,login_method,failed_login_count,is_new_device,event_velocity
0,2026-01-12 14:00:00,CASH_IN,330218.42,mobile,Android,Chrome,132.131.111.179,Bangalore,otp,2,1,1.26
1,2026-01-01 15:00:00,PAYMENT,11647.08,mobile,Android,Edge,24.222.32.170,Mumbai,otp,1,1,1.61
2,2026-01-01 10:00:00,CASH_IN,152264.21,desktop,Android,Edge,150.68.241.98,Mumbai,password,1,1,2.25
3,2026-01-17 19:00:00,TRANSFER,1551760.63,desktop,iOS,Safari,53.47.201.11,Pune,password,0,0,2.19
4,2026-01-09 14:00:00,CASH_IN,78172.3,mobile,Windows,Chrome,140.110.55.26,Delhi,password,1,0,0.48


In [47]:
import numpy as np
import pandas as pd

# 1. Convert to numeric
parsed_df['amount'] = pd.to_numeric(parsed_df['amount'], errors='coerce')
parsed_df['failed_login_count'] = pd.to_numeric(parsed_df['failed_login_count'], errors='coerce')
parsed_df['event_velocity'] = pd.to_numeric(parsed_df['event_velocity'], errors='coerce')
parsed_df['is_new_device'] = pd.to_numeric(parsed_df['is_new_device'], errors='coerce')

# 2. Add Log Transformation to amount (CRITICAL to lower ROC-AUC)
# This makes it harder for the model to "cheat" on high amounts
parsed_df['amount'] = np.log1p(parsed_df['amount'].fillna(0))

# 3. Timestamp features
parsed_df['timestamp'] = pd.to_datetime(parsed_df['timestamp'], errors='coerce')
parsed_df['event_hour'] = parsed_df['timestamp'].dt.hour
parsed_df['event_day'] = parsed_df['timestamp'].dt.day
parsed_df['event_month'] = parsed_df['timestamp'].dt.month

In [48]:
parsed_df = parsed_df.fillna({
    'amount': 0,
    'failed_login_count': 0,
    'event_velocity': 0,
    'is_new_device': 0,
    'device_type': 'unknown',
    'os': 'unknown',
    'browser': 'unknown',
    'ip_city': 'unknown',
    'login_method': 'unknown'
})


In [49]:
parsed_df.shape

(100000, 15)

In [50]:
parsed_df['isFraud'] = df['isFraud']

# Drop the bad row
parsed_df = parsed_df.dropna(subset=['isFraud'])

In [51]:
numeric_cols = [
    'amount',
    'failed_login_count',
    'event_velocity',
    'is_new_device',
    'event_hour',
    'event_day',
    'event_month'
]

categorical_cols = [
    'transaction_type',
    'device_type',
    'os',
    'browser',
    'ip_city',
    'login_method'
]


In [52]:
parsed_df['isFraud'].isnull().sum()

np.int64(0)

In [53]:
from sklearn.model_selection import train_test_split

X = parsed_df.drop(columns=['isFraud', 'timestamp', 'ip_address'])
y = parsed_df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


Autoencoder

In [54]:
# Use ONLY normal transactions
X_train_normal = X_train[y_train == 0]

In [55]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train_normal)
X_test_processed = preprocessor.transform(X_test)

In [56]:
import tensorflow as tf
from tensorflow.keras import layers, models

# The input_dim is determined by your processed features
input_dim = X_train_processed.shape[1]

# We are tightening the bottleneck from 32 to 8 or 16
autoencoder = models.Sequential([
    # Encoder
    layers.Dense(32, activation='relu', input_shape=(input_dim,)),
    layers.Dense(16, activation='relu'),

    # Bottleneck (The "Essence" Layer)
    layers.Dense(8, activation='relu'),

    # Decoder
    layers.Dense(16, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(input_dim, activation='linear')
])

autoencoder.compile(optimizer='adam', loss='mse')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [57]:
autoencoder.fit(
    X_train_processed, X_train_processed,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

Epoch 1/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.2659 - val_loss: 0.2016
Epoch 2/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1504 - val_loss: 0.1246
Epoch 3/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1201 - val_loss: 0.1158
Epoch 4/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1128 - val_loss: 0.1112
Epoch 5/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1090 - val_loss: 0.1080
Epoch 6/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1064 - val_loss: 0.1062
Epoch 7/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1052 - val_loss: 0.1053
Epoch 8/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1044 - val_loss: 0.1045
Epoch 9/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1040 - val_loss: 0.1042
Epoch 10/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1037 - val_loss: 0.1039
Epoch 11/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1035 - val_loss: 0.1039
Epoch 12/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

In [58]:
reconstructions = autoencoder.predict(X_test_processed)

import numpy as np
mse = np.mean(np.power(X_test_processed - reconstructions, 2), axis=1)

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [59]:
threshold = np.percentile(mse, 95)  # tune this

ae_preds = (mse > threshold).astype(int)


In [60]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

print("AUTOENCODER RESULTS ")
print(classification_report(y_test, ae_preds))
print("ROC-AUC:", roc_auc_score(y_test, mse))
print("PR-AUC:", average_precision_score(y_test, mse))


AUTOENCODER RESULTS 🔥
              precision    recall  f1-score   support

           0       1.00      0.95      0.97     19972
           1       0.02      0.57      0.03        28

    accuracy                           0.95     20000
   macro avg       0.51      0.76      0.50     20000
weighted avg       1.00      0.95      0.97     20000

ROC-AUC: 0.8257793053131527
PR-AUC: 0.127348070252868


Random Forest

In [61]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42
    ))
])

model.fit(X_train, y_train)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['amount',
                                                   'failed_login_count',
                                                   'event_velocity',
                                                   'is_new_device',
                                                   'event_hour', 'event_day',
                                                   'event_month']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['transaction_type',
                                                   'device_type', 'os',
                                                   'browser', 'ip_city',
                                                   'login_method'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=200, random_state=42))])

In [62]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print("SIEM PIPELINE RESULTS ")
print(classification_report(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, probs))
print("PR-AUC:", average_precision_score(y_test, probs))


SIEM PIPELINE RESULTS 🔥
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19972
           1       1.00      0.82      0.90        28

    accuracy                           1.00     20000
   macro avg       1.00      0.91      0.95     20000
weighted avg       1.00      1.00      1.00     20000

ROC-AUC: 0.9999982117822095
PR-AUC: 0.9987684729064039


XGBoost

In [63]:
!pip install xgboost

In [64]:
from xgboost import XGBClassifier

xgb_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', XGBClassifier(
        n_estimators=50,       # Lowered from 200
        max_depth=2,           # Very shallow trees
        gamma=10,              # High penalty for making new splits
        reg_alpha=20,          # High L1 regularization
        learning_rate=0.01,    # Very slow learning
        scale_pos_weight=(len(y_train[y_train==0]) / len(y_train[y_train==1])),
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['amount',
                                                   'failed_login_count',
                                                   'event_velocity',
                                                   'is_new_device',
                                                   'event_hour', 'event_day',
                                                   'event_month']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['transaction_type',
                                                   'device_type', 'os',
                                                   'browser', 'ip_city',
                                                   'login_method'])])),
                ('model',
                 XGBClassifier(base_sco...
                               feature_types=None, feature_weights=None,
                               gamma=10, grow_policy=None, importance_type=None,
                               interaction_constraints=None, learning_rate=0.01,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=2, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=50, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [65]:
xgb_preds = xgb_model.predict(X_test)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

print("XGBOOST RESULTS ")
print(classification_report(y_test, xgb_preds))
print("ROC-AUC:", roc_auc_score(y_test, xgb_probs))
print("PR-AUC:", average_precision_score(y_test, xgb_probs))

XGBOOST RESULTS 🔥
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     19972
           1       0.11      0.96      0.19        28

    accuracy                           0.99     20000
   macro avg       0.55      0.98      0.59     20000
weighted avg       1.00      0.99      0.99     20000

ROC-AUC: 0.9930921146748305
PR-AUC: 0.25443703087786557


Score for anamoly and fraud

In [66]:
import numpy as np

# ✅ Use SAME preprocessing (DO NOT refit)
X_train_normal = X_train[y_train == 0]

# Transform using already fitted preprocessor (from RF/XGB pipeline)
X_train_proc = preprocessor.transform(X_train_normal)
X_test_proc  = preprocessor.transform(X_test)

# --- Reconstruction ---
recon_train = autoencoder.predict(X_train_proc)
recon_test  = autoencoder.predict(X_test_proc)

# --- Reconstruction error (MSE) ---
mse_train = np.mean((X_train_proc - recon_train) ** 2, axis=1)
mse_test  = np.mean((X_test_proc  - recon_test)  ** 2, axis=1)

# --- Normalize (VERY IMPORTANT for combining) ---
ae_score_train = (mse_train - mse_train.min()) / (mse_train.max() - mse_train.min() + 1e-8)
ae_score_test  = (mse_test  - mse_test.min())  / (mse_test.max()  - mse_test.min()  + 1e-8)


2497/2497 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [67]:
import numpy as np


ae_norm = (ae_score_test - ae_score_test.min()) / (ae_score_test.max() - ae_score_test.min() + 1e-8)

final_score = (
    0.5 * ae_norm +      # Increased weight: Focus more on behavioral anomaly
    0.25 * probs +       # Reduced weight: Random Forest probability
    0.25 * xgb_probs     # Reduced weight: XGBoost probability
)

threshold = np.percentile(final_score, 97)
final_preds = (final_score > threshold).astype(int)

print("HYBRID RISK FUSION COMPLETE ")

HYBRID RISK FUSION COMPLETE 🔥


In [68]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

print("HYBRID MODEL RESULTS ")
print(classification_report(y_test, final_preds))
print("ROC-AUC:", roc_auc_score(y_test, final_score))
print("PR-AUC:", average_precision_score(y_test, final_score))


HYBRID MODEL RESULTS 🔥
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     19972
           1       0.05      1.00      0.09        28

    accuracy                           0.97     20000
   macro avg       0.52      0.99      0.54     20000
weighted avg       1.00      0.97      0.98     20000

ROC-AUC: 0.9989681983348115
PR-AUC: 0.7841705391293161
